# Synthetic Comment Generation for r/AskDocs

This notebook generates synthetic comments for each OP in the r/AskDocs corpus using
Google's Gemini 2.5 Pro. For each submission, three sets of synthetic comments are generated:

1. **75/25 clinician-to-layperson** ratio
2. **25/75 clinician-to-layperson** ratio
3. **No specified ratio** (model decides)

The number of synthetic comments per OP matches the number of real top-level comments.

**Output**: A single JSONL file where each line contains the OP, its real comments, and all
three sets of synthetic comments.

## 1. Setup & Imports

In [3]:
# Install dependencies if needed
# !pip install google-genai tqdm

In [1]:
import json
import os
import time
from getpass import getpass
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

from google import genai
from google.genai import types

In [2]:
# ---------- Configuration ----------
API_KEY = getpass("Enter your Google Gemini API key: ")
MODEL_ID = "gemini-2.5-pro"

# Paths
CORPORA_DIR = Path("../output/corpora")
SUBMISSIONS_PATH = CORPORA_DIR / "submissions_corpus.jsonl"
COMMENTS_PATH = CORPORA_DIR / "comments_corpus.jsonl"
CHECKPOINT_DIR = CORPORA_DIR / "checkpoints"
OUTPUT_PATH = CORPORA_DIR / "synthetic_comments_corpus.jsonl"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Rate-limiting
DELAY_BETWEEN_CALLS = 1.0  # seconds between API calls (can be adjusted based on quota)
CHECKPOINT_EVERY = 50      # save progress every N submissions via checkpointing

print(f"Model: {MODEL_ID}")
print(f"Submissions: {SUBMISSIONS_PATH}")
print(f"Comments: {COMMENTS_PATH}")
print(f"Output: {OUTPUT_PATH}")

Enter your Google Gemini API key:  ········


Model: gemini-2.5-pro
Submissions: ../output/corpora/submissions_corpus.jsonl
Comments: ../output/corpora/comments_corpus.jsonl
Output: ../output/corpora/synthetic_comments_corpus.jsonl


In [3]:
# Initialize the Gemini client
client = genai.Client(api_key=API_KEY)
print("Gemini client initialized.")

Gemini client initialized.


## 2. Load Data & Compute Top-Level Comment Counts

In [4]:
# Load all submissions
submissions = []
with open(SUBMISSIONS_PATH) as f:
    for line in f:
        submissions.append(json.loads(line))

print(f"Loaded {len(submissions):,} submissions")

# Build a lookup by submission id
submissions_by_id = {s["id"]: s for s in submissions}

Loaded 19,984 submissions


In [5]:
# Load all comments and group by submission
comments_by_submission = defaultdict(list)

with open(COMMENTS_PATH) as f:
    for line in f:
        comment = json.loads(line)
        # Top-level comments have parent_id starting with t3_ (link to submission)
        # Threaded replies have parent_id starting with t1_ (link to another comment)
        if comment.get("parent_id", "").startswith("t3_"):
            # Extract the submission id from parent_id (e.g., t3_1nutdta -> 1nutdta)
            sub_id = comment["parent_id"][3:]
            comments_by_submission[sub_id].append(comment)

print(f"Found top-level comments for {len(comments_by_submission):,} submissions")
print(f"Total top-level comments: {sum(len(v) for v in comments_by_submission.values()):,}")

Found top-level comments for 19,984 submissions
Total top-level comments: 40,537


In [6]:
# Verify against the pre-computed total_top_level_comments field
mismatches = 0
for sub in submissions[:100]:  # spot-check first 100
    precomputed = sub.get("total_top_level_comments", 0)
    actual = len(comments_by_submission.get(sub["id"], []))
    if precomputed != actual:
        mismatches += 1

print(f"Mismatches in first 100 submissions: {mismatches}")
if mismatches == 0:
    print("Counts match the pre-computed field.")

Mismatches in first 100 submissions: 0
Counts match the pre-computed field.


In [7]:
# Distribution of top-level comment counts
import statistics

counts = [len(comments_by_submission.get(s["id"], [])) for s in submissions]
print(f"Min: {min(counts)}, Max: {max(counts)}, Median: {statistics.median(counts)}, Mean: {statistics.mean(counts):.1f}")
print(f"Submissions with 0 top-level comments: {counts.count(0):,}")
print(f"Submissions with >=1 top-level comment: {len([c for c in counts if c >= 1]):,}")

Min: 1, Max: 127, Median: 1.0, Mean: 2.0
Submissions with 0 top-level comments: 0
Submissions with >=1 top-level comment: 19,984


## 3. Prompt Templates

In [8]:
PROMPT_TEMPLATES = {
    "75_clinician": (
        "The following is the opening post from a thread in the subreddit r/AskDocs. "
        "Generate {n} comments in which each comment responds to the opening post and "
        "any subsequent comments prior to it, i.e. none of the comments should be threaded. "
        "Of these comments, 75% should imitate the response of a clinician and 25% should "
        "imitate a comment from a user with no medical training.\n\n"
        "Return your response as a JSON array of objects, where each object has these fields:\n"
        "- \"body\": the text of the comment\n"
        "- \"author_type\": either \"clinician\" or \"layperson\"\n\n"
        "Opening Post:\n"
        "Title: {title}\n\n"
        "{selftext}"
    ),
    "25_clinician": (
        "The following is the opening post from a thread in the subreddit r/AskDocs. "
        "Generate {n} comments in which each comment responds to the opening post and "
        "any subsequent comments prior to it, i.e. none of the comments should be threaded. "
        "Of these comments, 25% should imitate the response of a clinician and 75% should "
        "imitate a comment from a user with no medical training.\n\n"
        "Return your response as a JSON array of objects, where each object has these fields:\n"
        "- \"body\": the text of the comment\n"
        "- \"author_type\": either \"clinician\" or \"layperson\"\n\n"
        "Opening Post:\n"
        "Title: {title}\n\n"
        "{selftext}"
    ),
    "no_ratio": (
        "The following is the opening post from a thread in the subreddit r/AskDocs. "
        "Generate {n} comments in which each comment responds to the opening post and "
        "any subsequent comments prior to it, i.e. none of the comments should be threaded.\n\n"
        "Return your response as a JSON array of objects, where each object has these fields:\n"
        "- \"body\": the text of the comment\n"
        "- \"author_type\": either \"clinician\" or \"layperson\" (use your best judgment)\n\n"
        "Opening Post:\n"
        "Title: {title}\n\n"
        "{selftext}"
    ),
}

CONDITION_KEYS = list(PROMPT_TEMPLATES.keys())
print(f"Prompt conditions: {CONDITION_KEYS}")

Prompt conditions: ['75_clinician', '25_clinician', 'no_ratio']


## 4. Gemini API Call Logic

In [9]:
def call_gemini(prompt: str, retries: int = 3) -> list[dict]:
    """
    Send a prompt to Gemini and parse the JSON array response.
    Returns a list of comment dicts with 'body' and 'author_type' fields.
    """
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=1.0,
                ),
            )
            # Parse the JSON response
            raw_text = response.text.strip()
            comments = json.loads(raw_text)

            if not isinstance(comments, list):
                raise ValueError(f"Expected a JSON array, got {type(comments).__name__}")

            # Validate structure
            validated = []
            for c in comments:
                validated.append({
                    "body": str(c.get("body", "")),
                    "author_type": str(c.get("author_type", "unknown")),
                })
            return validated

        except json.JSONDecodeError as e:
            print(f"  JSON parse error (attempt {attempt+1}/{retries}): {e}")
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
        except Exception as e:
            print(f"  API error (attempt {attempt+1}/{retries}): {e}")
            if attempt < retries - 1:
                time.sleep(2 ** attempt)

    # All retries failed
    print(f"  WARNING: All {retries} retries exhausted. Returning empty list.")
    return []


def generate_synthetic_comments(submission: dict, n_comments: int) -> dict:
    """
    Generate all three sets of synthetic comments for a single submission.
    Returns a dict keyed by condition name, each containing a list of comment dicts.
    """
    results = {}
    title = submission.get("title", "")
    selftext = submission.get("selftext", "")

    for condition_key, template in PROMPT_TEMPLATES.items():
        prompt = template.format(n=n_comments, title=title, selftext=selftext)
        comments = call_gemini(prompt)
        results[condition_key] = comments
        time.sleep(DELAY_BETWEEN_CALLS)

    return results


print("API functions defined.")

API functions defined.


## 5. Checkpoint Utilities

In [10]:
CHECKPOINT_FILE = CHECKPOINT_DIR / "generation_progress.jsonl"
CHECKPOINT_INDEX_FILE = CHECKPOINT_DIR / "completed_ids.json"


def load_checkpoint() -> tuple[dict, set]:
    """
    Load previously completed results from the checkpoint file.
    Returns (results_by_id, completed_ids).
    """
    results_by_id = {}
    completed_ids = set()

    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE) as f:
            for line in f:
                record = json.loads(line)
                sub_id = record["submission_id"]
                results_by_id[sub_id] = record
                completed_ids.add(sub_id)
        print(f"Resumed from checkpoint: {len(completed_ids):,} submissions already completed.")
    else:
        print("No checkpoint found. Starting from scratch.")

    return results_by_id, completed_ids


def append_checkpoint(record: dict):
    """Append a single completed record to the checkpoint file."""
    with open(CHECKPOINT_FILE, "a") as f:
        f.write(json.dumps(record) + "\n")


print(f"Checkpoint file: {CHECKPOINT_FILE}")

Checkpoint file: ../output/corpora/checkpoints/generation_progress.jsonl


## 6. Main Generation Loop

In [11]:
# Load any existing progress
results_by_id, completed_ids = load_checkpoint()

# Filter to submissions that have at least 1 top-level comment
eligible_submissions = [
    s for s in submissions
    if len(comments_by_submission.get(s["id"], [])) >= 1
]
remaining = [s for s in eligible_submissions if s["id"] not in completed_ids]

print(f"Total eligible submissions (>=1 top-level comment): {len(eligible_submissions):,}")
print(f"Already completed: {len(completed_ids):,}")
print(f"Remaining: {len(remaining):,}")
print(f"API calls needed: ~{len(remaining) * 3:,}")

Resumed from checkpoint: 2 submissions already completed.
Total eligible submissions (>=1 top-level comment): 19,984
Already completed: 2
Remaining: 19,982
API calls needed: ~59,946


In [14]:
# ---------- Run generation ----------
import random

errors = []
sample = random.sample(remaining, min(1, len(remaining)))

for i, sub in enumerate(tqdm(sample, desc="Generating synthetic comments")):
    sub_id = sub["id"]
    n_top_level = len(comments_by_submission.get(sub_id, []))

    if n_top_level == 0:
        continue

    try:
        synthetic = generate_synthetic_comments(sub, n_top_level)

        record = {
            "submission_id": sub_id,
            "n_top_level_comments": n_top_level,
            "synthetic_75_clinician": synthetic["75_clinician"],
            "synthetic_25_clinician": synthetic["25_clinician"],
            "synthetic_no_ratio": synthetic["no_ratio"],
        }

        results_by_id[sub_id] = record
        completed_ids.add(sub_id)
        append_checkpoint(record)

    except Exception as e:
        errors.append((sub_id, str(e)))
        print(f"\nError on submission {sub_id}: {e}")

print(f"\nGeneration complete. Processed {len(completed_ids):,} submissions.")
if errors:
    print(f"Errors encountered: {len(errors)}")
    for sub_id, err in errors[:10]:
        print(f"  {sub_id}: {err}")

Generating synthetic comments: 100%|██████████████| 1/1 [00:36<00:00, 36.69s/it]


Generation complete. Processed 5 submissions.


In [13]:
# Look at real vs. synthetic comments for this small sample

condition_labels = {
    "synthetic_75_clinician": "75% of comments from a clinician",
    "synthetic_25_clinician": "25% of comments from a clinician",
    "synthetic_no_ratio": "No specified clinician/layperson ratio",
}

for sub_id, record in results_by_id.items():
    sub = submissions_by_id[sub_id]
    real = comments_by_submission[sub_id]
    
    print(f"=== Opening Post: {sub['title']} ===\n")
    
    print("-- Real comments --")
    for c in sorted(real, key=lambda x: x.get("created_utc", 0))[:3]:
        flair = c.get("author_flair_text") or "No flair"
        print(f"  [{flair}] {c['body']}\n")
    
    for condition, label in condition_labels.items():
        print(f"-- {label} --")
        for c in record[condition][:3]:
            print(f"  [{c['author_type']}] {c['body']}\n")
    
    print("\n")

=== Opening Post: How would I go about getting an amputation that is not necessary? ===

-- Real comments --
  [No flair] [removed]

  [Registered Nurse] This would absolutely violate medical ethics and no licensed physician would do this.

You’ll do better to get psychiatric treatment for the desire to do this

  [No flair] [removed]

-- 75% of comments from a clinician --
  [clinician] Thank you for reaching out with your question. To answer directly, no, a surgeon will not perform an elective amputation of a healthy limb. This is fundamentally different from other body modifications as it involves the removal of a functional, healthy body part, which goes against the core medical principle of 'first, do no harm.'

The feelings you're describing, a persistent desire to be an amputee, are characteristic of a condition known as Body Integrity Dysphoria (BID), formerly called Body Integrity Identity Disorder (BIID). This is a complex neuropsychiatric condition where there is a mismatch 

## 7. Assemble Final Output JSONL

In [ ]:
# Build the final combined JSONL
# Each line: OP + real comments + 3 sets of synthetic comments

n_written = 0

with open(OUTPUT_PATH, "w") as out:
    for sub in tqdm(submissions, desc="Assembling output"):
        sub_id = sub["id"]
        real_comments = comments_by_submission.get(sub_id, [])
        synthetic = results_by_id.get(sub_id)

        # Only include submissions that have both real and synthetic comments
        if not real_comments or synthetic is None:
            continue

        record = {
            # ---- Submission (OP) ----
            "submission": {
                "id": sub["id"],
                "title": sub.get("title", ""),
                "selftext": sub.get("selftext", ""),
                "author": sub.get("author", ""),
                "author_flair_text": sub.get("author_flair_text"),
                "score": sub.get("score"),
                "num_comments": sub.get("num_comments"),
                "total_top_level_comments": sub.get("total_top_level_comments"),
                "created_utc": sub.get("created_utc"),
                "permalink": sub.get("permalink"),
                "subreddit": sub.get("subreddit"),
                "layperson_comment_fraction": sub.get("layperson_comment_fraction"),
            },
            # ---- Real comments ----
            "real_comments": [
                {
                    "id": c["id"],
                    "body": c.get("body", ""),
                    "author": c.get("author", ""),
                    "author_flair_text": c.get("author_flair_text"),
                    "score": c.get("score"),
                    "created_utc": c.get("created_utc"),
                    "is_submitter": c.get("is_submitter", False),
                }
                for c in sorted(real_comments, key=lambda x: x.get("created_utc", 0))
            ],
            # ---- Synthetic comments: 75% clinician / 25% layperson ----
            "synthetic_75_clinician": synthetic["synthetic_75_clinician"],
            # ---- Synthetic comments: 25% clinician / 75% layperson ----
            "synthetic_25_clinician": synthetic["synthetic_25_clinician"],
            # ---- Synthetic comments: no specified ratio ----
            "synthetic_no_ratio": synthetic["synthetic_no_ratio"],
        }

        out.write(json.dumps(record) + "\n")
        n_written += 1

print(f"\nWrote {n_written:,} records to {OUTPUT_PATH}")

## 8. Validation

In [ ]:
# Quick validation: load the output and spot-check
sample_records = []
with open(OUTPUT_PATH) as f:
    for i, line in enumerate(f):
        if i < 5:
            sample_records.append(json.loads(line))
        else:
            break

for rec in sample_records[:2]:
    sub = rec["submission"]
    print(f"--- Submission: {sub['id']} ---")
    print(f"  Title: {sub['title'][:80]}")
    print(f"  Real comments: {len(rec['real_comments'])}")
    print(f"  Synthetic 75% clinician: {len(rec['synthetic_75_clinician'])} comments")
    print(f"  Synthetic 25% clinician: {len(rec['synthetic_25_clinician'])} comments")
    print(f"  Synthetic no ratio: {len(rec['synthetic_no_ratio'])} comments")

    # Show a sample synthetic comment
    if rec["synthetic_75_clinician"]:
        sample = rec["synthetic_75_clinician"][0]
        print(f"  Sample (75% clin): [{sample['author_type']}] {sample['body'][:100]}...")
    print()

In [ ]:
# Summary statistics on the output file
total_records = 0
total_real = 0
total_syn_75 = 0
total_syn_25 = 0
total_syn_nr = 0

with open(OUTPUT_PATH) as f:
    for line in f:
        rec = json.loads(line)
        total_records += 1
        total_real += len(rec["real_comments"])
        total_syn_75 += len(rec["synthetic_75_clinician"])
        total_syn_25 += len(rec["synthetic_25_clinician"])
        total_syn_nr += len(rec["synthetic_no_ratio"])

print(f"Output records: {total_records:,}")
print(f"Total real comments: {total_real:,}")
print(f"Total synthetic (75% clinician): {total_syn_75:,}")
print(f"Total synthetic (25% clinician): {total_syn_25:,}")
print(f"Total synthetic (no ratio): {total_syn_nr:,}")